# Water Potability Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify water samples as **potable (safe to drink)** or not based on 9 chemical properties: pH, hardness, solids, chloramines, sulfate, conductivity, organic carbon, trihalomethanes, and turbidity. Synthetic dataset with ~3,200 rows and ~39% potable rate.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given 9 water quality measurements (pH, hardness, solids, chloramines, sulfate, conductivity, organic carbon, trihalomethanes, turbidity), classify whether the water is potable (1) or not (0).

## 5 · Why This Project Matters

- **Clean water access** affects 2 billion people worldwide.
- Automated water quality testing enables faster, cheaper monitoring.
- This dataset is notoriously **hard** — near-random class separation challenges all models.
- It teaches when ML **cannot** solve a problem well due to dataset limitations.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~3,200 |
| Features | 9 (ph, Hardness, Solids, Chloramines, Sulfate, Conductivity, Organic_carbon, Trihalomethanes, Turbidity) |
| Target | `Potability` (binary: 0=not potable, 1=potable) |
| Class balance | ~61% not potable, ~39% potable |
| Missing values | Injected in ph, Sulfate, Trihalomethanes |

## 7 · Dataset Source and License Notes

**Source**: Water Potability dataset (Kaggle).
**License**: Public / educational.
**Note**: Synthetic water quality measurements.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "Potability"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

In [ ]:
np.random.seed(SEED)
n = 3276

ph = np.random.normal(7.1, 1.6, n).clip(0, 14).round(4)
hardness = np.random.normal(196, 33, n).clip(47, 324).round(4)
solids = np.random.normal(22000, 8500, n).clip(320, 56500).round(4)
chloramines = np.random.normal(7.1, 1.6, n).clip(0.4, 13.1).round(4)
sulfate = np.random.normal(333, 41, n).clip(129, 481).round(4)
conductivity = np.random.normal(426, 81, n).clip(181, 753).round(4)
organic_carbon = np.random.normal(14.3, 3.3, n).clip(2.2, 28.3).round(4)
trihalomethanes = np.random.normal(66.4, 16.2, n).clip(0.7, 124).round(4)
turbidity = np.random.normal(3.97, 0.78, n).clip(1.5, 6.7).round(4)

# Weak signal (this dataset is intentionally hard)
score = (
    0.05 * (ph - 7) ** 2
    + 0.002 * (hardness - 200)
    - 0.00001 * solids
    + 0.03 * chloramines
    - 0.002 * sulfate
    + 0.001 * conductivity
    + 0.02 * organic_carbon
    - 0.005 * trihalomethanes
    + 0.05 * turbidity
    + np.random.normal(0, 1.2, n)
)
potability = (score > 0.5).astype(int)

df = pd.DataFrame({
    'ph': ph, 'Hardness': hardness, 'Solids': solids, 'Chloramines': chloramines,
    'Sulfate': sulfate, 'Conductivity': conductivity, 'Organic_carbon': organic_carbon,
    'Trihalomethanes': trihalomethanes, 'Turbidity': turbidity, 'Potability': potability,
})

# Inject some missing values like the original dataset
for col in ['ph', 'Sulfate', 'Trihalomethanes']:
    mask = np.random.random(n) < 0.1
    df.loc[mask, col] = np.nan

print(f"Dataset shape: {df.shape}")
print(f"Potability rate: {df['Potability'].mean():.2%}")
print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for i, col in enumerate(num_cols[:9]):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=25, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=10)
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for i, col in enumerate(num_cols[:9]):
    ax = axes[i // 3, i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col, fontsize=10)
plt.suptitle("Features by Potability", fontsize=14)
plt.tight_layout()
plt.show()
print(f"Features: {len(num_cols)}")

## 14 · Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['coral', 'steelblue'], edgecolor='black')
axes[0].set_title("Potability Distribution")
axes[0].set_xticklabels(['Not Potable (0)', 'Potable (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['coral', 'steelblue'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")

## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [ ]:
# Impute missing values with median
for col in df.columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [ ]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Logistic Regression as baseline.

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [ ]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

## 25 · Interpretation and Business Insight

- This dataset is **notoriously difficult** — features barely separate classes.
- No single feature strongly predicts potability.
- Models struggle to exceed ~65-70% accuracy (barely above the 61% baseline).
- This teaches a critical lesson: **not every problem is solvable with the given features**.
- Real water quality testing uses precise thresholds, not ML on noisy distributions.

## 26 · Limitations

1. **Very weak signal** — features overlap almost completely between classes.
2. Missing values in key features (pH, sulfate, trihalomethanes).
3. No feature interactions or domain thresholds modeled.
4. Synthetic data — real water quality has strict WHO/EPA thresholds.
5. Binary classification loses nuance (multiple contaminant types).

## 27 · How to Improve This Project

1. Apply WHO/EPA regulatory thresholds as hard rules.
2. Engineer interaction features (pH × chloramines).
3. Use more advanced imputation (KNN, iterative).
4. Collect more relevant features (bacteria counts, metal concentrations).
5. Use anomaly detection instead of binary classification.

## 28 · Production Considerations

- Water quality monitoring must use **regulatory thresholds**, not ML predictions.
- ML can supplement but not replace chemical testing.
- Real-time sensor data with drift detection.
- Safety-critical — false positives (saying unsafe water is safe) are dangerous.

## 29 · Common Mistakes

1. Expecting high accuracy on a near-random dataset.
2. Not checking if features actually separate classes.
3. Using accuracy on imbalanced data.
4. Not comparing to the majority-class baseline (61%).
5. Deploying ML for safety-critical decisions without validation.

## 30 · Mini Challenge / Exercises

1. Calculate the majority-class baseline accuracy.
2. Apply WHO pH thresholds (6.5-8.5) as a rule-based classifier.
3. Try different imputation strategies — does accuracy change?
4. Build a model with only pH and turbidity — how close to full model?

## 31 · Final Summary / Key Takeaways

1. Water potability is a hard problem with weak feature signals.
2. Models struggle to beat the majority-class baseline significantly.
3. This teaches when ML is **not** the right approach.
4. Real water quality relies on regulatory thresholds, not ML.
5. Safety-critical applications need domain rules, not just predictions.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))